## BERTScore evaluation — all QA approaches vs gold answers

Compares predicted answers from `qa_eval_*_outputs.jsonl` against `gold_answer` in `qa_eval_dataset.jsonl` using [BERTScore](https://github.com/Tiiiger/bert_score).

Set `APPROACH` to one key in `APPROACH_REGISTRY`, or `RUN_ALL_APPROACHES = True` to score every system.

In [ ]:
# bert-score 0.3.x requires transformers 4.x (breaks on transformers 5.x)
import importlib.util
import subprocess
import sys

INSTALL_ARGS = ['bert-score', 'transformers>=4.44,<5']

need_install = importlib.util.find_spec('bert_score') is None
try:
    import transformers

    if int(transformers.__version__.split('.')[0]) >= 5:
        need_install = True
        print(f'Incompatible transformers {transformers.__version__}; pinning to 4.x')
except ImportError:
    need_install = True

if need_install:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *INSTALL_ARGS])
else:
    import transformers as _tf
    print(f'bert-score ready (transformers {_tf.__version__})')

In [ ]:
from pathlib import Path
import json
import re

project_root = Path('.').resolve()
gold_path = project_root / 'qa_eval_dataset.jsonl'

APPROACH = 'graphdb'  # graphdb | rag_with_events | rag_no_events | hybrid | cdf_json_rag
RUN_ALL_APPROACHES = True

APPROACH_REGISTRY = {
    'graphdb': 'qa_eval_graphdb_outputs.jsonl',
    'rag_with_events': 'qa_eval_rag_with_events_outputs.jsonl',
    'rag_no_events': 'qa_eval_rag_no_events_outputs.jsonl',
    'hybrid': 'qa_eval_hybrid_outputs.jsonl',
    'cdf_json_rag': 'qa_eval_cdf_json_rag_outputs.jsonl',
}

# BERTScore settings (see https://github.com/Tiiiger/bert_score)
MODEL_TYPE = 'roberta-large'
LANG = 'en'
RESCALE_WITH_BASELINE = True
BATCH_SIZE = 16

print('Gold:', gold_path.exists())
print('Approaches:', list(APPROACH_REGISTRY.keys()))
for key, fname in APPROACH_REGISTRY.items():
    exists = (project_root / fname).exists()
    print(f'  {key}: {fname} ({"found" if exists else "missing"})')

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    rows = []
    if not path.exists():
        return rows
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if line:
            rows.append(json.loads(line))
    return rows


def normalize_text(text: str) -> str:
    text = (text or '').strip()
    return re.sub(r'\s+', ' ', text)


def normalize_yes_no(text: str) -> str:
    t = normalize_text(text).rstrip('.').lower()
    if t in {'yes', 'no'}:
        return t.capitalize()
    return normalize_text(text)


def build_aligned_pairs(approach: str, pred_path: Path) -> list[dict]:
    gold_rows = load_jsonl(gold_path)
    pred_rows = load_jsonl(pred_path)
    pred_by_question = {row['question']: row for row in pred_rows}
    pred_by_index = {
        int(row['question_index']): row
        for row in pred_rows
        if row.get('question_index') is not None
    }

    pairs: list[dict] = []
    missing_predictions = []

    for i, gold in enumerate(gold_rows, start=1):
        pred = pred_by_question.get(gold['question']) or pred_by_index.get(i)
        if pred is None:
            missing_predictions.append(gold.get('id', f'row_{i}'))
            continue

        ref = gold.get('gold_answer', '')
        if gold.get('answer_type') == 'yes_no':
            ref = normalize_yes_no(ref)
        resp = pred.get('answer', '')
        if gold.get('answer_type') == 'yes_no':
            resp = normalize_yes_no(resp)

        pairs.append({
            'question_index': pred.get('question_index', i),
            'id': gold.get('id'),
            'template_id': gold.get('template_id'),
            'category': gold.get('category'),
            'answer_type': gold.get('answer_type'),
            'scope': gold.get('scope'),
            'question': gold['question'],
            'gold_answer': normalize_text(ref),
            'predicted_answer': normalize_text(resp),
            'approach': approach,
            'qa_error': pred.get('error'),
        })

    if missing_predictions:
        print(f'  Missing predictions: {len(missing_predictions)}')
    return pairs

In [ ]:
import os

# Avoid OpenMP crash when pip torch and conda MKL coexist (Windows + Anaconda).
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

import torch
from bert_score import BERTScorer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('Loading BERTScorer:', MODEL_TYPE)

scorer = BERTScorer(
    model_type=MODEL_TYPE,
    lang=LANG,
    rescale_with_baseline=RESCALE_WITH_BASELINE,
    device=DEVICE,
    batch_size=BATCH_SIZE,
)

In [ ]:
def mean(values: list[float]) -> float:
    return sum(values) / len(values) if values else 0.0


def summarize_group(rows: list[dict]) -> dict:
    return {
        'count': len(rows),
        'mean_precision': mean([r['bertscore_precision'] for r in rows]),
        'mean_recall': mean([r['bertscore_recall'] for r in rows]),
        'mean_f1': mean([r['bertscore_f1'] for r in rows]),
    }


def run_bertscore_for_approach(approach: str) -> dict:
    pred_file = APPROACH_REGISTRY[approach]
    pred_path = project_root / pred_file
    output_path = project_root / f'qa_eval_bertscore_{approach}_results.jsonl'
    summary_path = project_root / f'qa_eval_bertscore_{approach}_summary.json'

    print(f'\n=== BERTScore: {approach} ===')
    print('Predictions:', pred_path)
    if not pred_path.exists():
        print('SKIP — prediction file missing')
        return {'approach': approach, 'skipped': True, 'reason': 'missing predictions'}

    pairs = build_aligned_pairs(approach, pred_path)
    if not pairs:
        print('SKIP — no aligned pairs')
        return {'approach': approach, 'skipped': True, 'reason': 'no pairs'}

    qa_errors = sum(1 for p in pairs if p.get('qa_error'))
    empty_preds = sum(1 for p in pairs if not p['predicted_answer'])
    print(f'Aligned: {len(pairs)} | QA errors: {qa_errors} | empty predictions: {empty_preds}')

    refs = [p['gold_answer'] for p in pairs]
    cands = [p['predicted_answer'] for p in pairs]

    print('Computing BERTScore...')
    precision, recall, f1 = scorer.score(cands, refs)
    precision = precision.tolist()
    recall = recall.tolist()
    f1 = f1.tolist()

    for pair, p, r, f in zip(pairs, precision, recall, f1):
        pair['bertscore_precision'] = p
        pair['bertscore_recall'] = r
        pair['bertscore_f1'] = f

    summary = {
        'approach': approach,
        'pred_file': pred_file,
        'model_type': MODEL_TYPE,
        'lang': LANG,
        'rescale_with_baseline': RESCALE_WITH_BASELINE,
        'device': DEVICE,
        'num_pairs': len(pairs),
        'qa_errors': qa_errors,
        'empty_predictions': empty_preds,
        'overall': summarize_group(pairs),
    }

    by_answer_type: dict[str, dict] = {}
    for answer_type in sorted({p['answer_type'] for p in pairs if p.get('answer_type')}):
        by_answer_type[answer_type] = summarize_group(
            [p for p in pairs if p.get('answer_type') == answer_type]
        )
    summary['by_answer_type'] = by_answer_type

    by_category: dict[str, dict] = {}
    for category in sorted({p['category'] for p in pairs if p.get('category')}):
        by_category[category] = summarize_group(
            [p for p in pairs if p.get('category') == category]
        )
    summary['by_category'] = by_category

    ok_pairs = [p for p in pairs if not p.get('qa_error')]
    summary['without_qa_errors'] = summarize_group(ok_pairs)

    with output_path.open('w', encoding='utf-8') as f:
        for row in pairs:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

    o = summary['overall']
    print(
        f'Mean P/R/F1: {o["mean_precision"]:.4f} / '
        f'{o["mean_recall"]:.4f} / {o["mean_f1"]:.4f}'
    )
    print(f'Wrote {output_path.name} and {summary_path.name}')
    return summary

In [ ]:
approaches_to_run = list(APPROACH_REGISTRY.keys()) if RUN_ALL_APPROACHES else [APPROACH]
all_summaries: list[dict] = []

for approach in approaches_to_run:
    summary = run_bertscore_for_approach(approach)
    all_summaries.append(summary)

if RUN_ALL_APPROACHES:
    compare_path = project_root / 'qa_eval_bertscore_all_approaches_summary.json'
    compare_path.write_text(json.dumps(all_summaries, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'\nWrote cross-approach summary to {compare_path.name}')
    print('\nComparison (mean F1):')
    for s in all_summaries:
        if s.get('skipped'):
            print(f'  {s["approach"]}: SKIP ({s.get("reason")})')
        else:
            f1 = s['overall']['mean_f1']
            print(f'  {s["approach"]}: F1={f1:.4f} (n={s["num_pairs"]})')

In [ ]:
# Lowest F1 examples for the last scored approach (quick sanity check)
inspect_approach = approaches_to_run[-1]
results_path = project_root / f'qa_eval_bertscore_{inspect_approach}_results.jsonl'
if results_path.exists():
    scored = load_jsonl(results_path)
    worst = sorted(scored, key=lambda r: r['bertscore_f1'])[:10]
    print(f'Worst F1 — {inspect_approach}')
    for row in worst:
        print(f"\nF1={row['bertscore_f1']:.4f} | {row.get('id')} | error={bool(row.get('qa_error'))}")
        print('Q:', row['question'][:120])
        print('Gold:', row['gold_answer'][:160])
        print('Pred:', row['predicted_answer'][:160])
else:
    print(f'No results file for {inspect_approach}')